In [2]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import xgboost as xgb

os.makedirs('../models/diabetes', exist_ok=True)

print("✅ Libraries loaded")
print(f"   XGBoost version: {xgb.__version__}")

✅ Libraries loaded
   XGBoost version: 3.2.0


In [3]:
print("Loading diabetes features...")
df = pd.read_csv('../data/processed/diabetes_features.csv')
print(f"✅ Loaded: {df.shape}")
print(f"\nLabel distribution:")
print(df['diabetes_risk'].value_counts())

Loading diabetes features...
✅ Loaded: (304448, 17)

Label distribution:
diabetes_risk
medium    158757
low        81713
high       63978
Name: count, dtype: int64


In [4]:
feature_cols = [
    'age', 'bmi', 'avg_glucose', 'sugar_variability',
    'avg_systolic', 'avg_diastolic', 'bp_variability',
    'avg_pulse', 'adherence_rate', 'missed_doses',
    'has_chronic_condition', 'allergies_count',
    'physical_activity_score', 'diet_quality_score',
    'stress_score', 'sleep_score'
]

X = df[feature_cols]
y = df['diabetes_risk']

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"✅ Features shape: {X.shape}")
print(f"✅ Label classes: {le.classes_}")
print(f"   0={le.classes_[0]}, 1={le.classes_[1]}, 2={le.classes_[2]}")

✅ Features shape: (304448, 16)
✅ Label classes: ['high' 'low' 'medium']
   0=high, 1=low, 2=medium


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train set: {X_train_scaled.shape}")
print(f"✅ Test set:  {X_test_scaled.shape}")
print(f"\nTrain label distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"   {le.classes_[u]}: {c:,} ({c/len(y_train)*100:.1f}%)")

✅ Train set: (243558, 16)
✅ Test set:  (60890, 16)

Train label distribution:
   high: 51,182 (21.0%)
   low: 65,370 (26.8%)
   medium: 127,006 (52.1%)


In [6]:
print("Training XGBoost model...")
print("This will take 2-3 minutes...")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    num_class=3,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=50
)

print("\n✅ Model trained successfully!")

Training XGBoost model...
This will take 2-3 minutes...
[0]	validation_0-mlogloss:0.95941
[50]	validation_0-mlogloss:0.07372
[100]	validation_0-mlogloss:0.04825
[150]	validation_0-mlogloss:0.04351
[200]	validation_0-mlogloss:0.04169
[250]	validation_0-mlogloss:0.04098
[299]	validation_0-mlogloss:0.04072

✅ Model trained successfully!


In [7]:
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')

print(f"📊 Model Performance:")
print(f"   Accuracy:  {accuracy*100:.2f}%")
print(f"   AUC Score: {auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

📊 Model Performance:
   Accuracy:  98.22%
   AUC Score: 0.9993

Classification Report:
              precision    recall  f1-score   support

        high       0.98      0.98      0.98     12796
         low       0.98      0.98      0.98     16343
      medium       0.98      0.98      0.98     31751

    accuracy                           0.98     60890
   macro avg       0.98      0.98      0.98     60890
weighted avg       0.98      0.98      0.98     60890



In [8]:
# Feature importance without charts
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("📊 Feature Importance:")
for _, row in importance_df.iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"   {row['feature']:<30} {bar} {row['importance']:.4f}")

print("\n📊 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"   Classes: {list(le.classes_)}")
print(cm)

📊 Feature Importance:
   avg_glucose                    █████████████████████████████████████████████████████████ 0.5703
   sugar_variability              ██████████████ 0.1460
   has_chronic_condition          ████████████ 0.1293
   bmi                            ███ 0.0311
   age                            ██ 0.0248
   stress_score                   █ 0.0175
   adherence_rate                 █ 0.0150
   bp_variability                 █ 0.0114
   missed_doses                   █ 0.0103
   avg_systolic                   █ 0.0102
   sleep_score                     0.0082
   physical_activity_score         0.0081
   diet_quality_score              0.0069
   avg_diastolic                   0.0050
   allergies_count                 0.0030
   avg_pulse                       0.0027

📊 Confusion Matrix:
   Classes: ['high', 'low', 'medium']
[[12549     2   245]
 [    1 16091   251]
 [  278   305 31168]]


In [9]:
# Save model and scaler
joblib.dump(model, '../models/diabetes/diabetes_model.pkl')
joblib.dump(scaler, '../models/diabetes/diabetes_scaler.pkl')
joblib.dump(le, '../models/diabetes/diabetes_label_encoder.pkl')

# Save feature metadata
metadata = {
    "features": feature_cols,
    "target": "diabetes_risk",
    "classes": list(le.classes_),
    "n_samples_train": int(X_train_scaled.shape[0]),
    "n_samples_test": int(X_test_scaled.shape[0]),
    "accuracy": round(float(accuracy), 4),
    "auc_score": round(float(auc), 4),
    "model_version": "1.0",
    "trained_on": pd.Timestamp.now().strftime("%Y-%m-%d")
}

with open('../models/diabetes/diabetes_features.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ All files saved to models/diabetes/:")
print("   diabetes_model.pkl")
print("   diabetes_scaler.pkl")
print("   diabetes_label_encoder.pkl")
print("   diabetes_features.json")
print(f"\n📊 Final Summary:")
print(f"   Accuracy:  {accuracy*100:.2f}%")
print(f"   AUC Score: {auc:.4f}")
print(f"   Classes:   {list(le.classes_)}")

✅ All files saved to models/diabetes/:
   diabetes_model.pkl
   diabetes_scaler.pkl
   diabetes_label_encoder.pkl
   diabetes_features.json

📊 Final Summary:
   Accuracy:  98.22%
   AUC Score: 0.9993
   Classes:   ['high', 'low', 'medium']
